In [2]:
import pytesseract
from pdf2image import convert_from_path
import base64
from PIL import Image
from dotenv import load_dotenv
import os
import io
from openai import OpenAI

# Load .env file
load_dotenv()

# Create client
client = OpenAI(api_key=os.getenv("LM_STUDIO_API_KEY"),
                base_url=os.getenv("LM_STUDIO_BASE_URL"))
LLM_VISION_MODEL = str(os.getenv("LM_STUDIO_VISION_MODEL"))

# Set paths
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Confidence Config
LOW_CONF_THRESHOLD = 30    # Confidence unter 30% = unsicher
LOW_CONF_MAX_COUNT = 2     # Mehr als 2 unsichere Wörter → Fallback

def image_to_base64(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

# Path to a scanned PDF
pdf_path = r"D:\Coding\Projects\content-aware-fileserver\data\samples\Wasserzählerstand_2026-01-21_122831.pdf"
# PDF → Images → Text
images = convert_from_path(pdf_path, poppler_path=r"C:\Program Files\poppler-25.12.0\Library\bin")
for image in images:
    text = pytesseract.image_to_string(image, lang="deu+eng")
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    scores = [int(c) for c in data['conf'] if int(c) > 0]
    low_confidence = [s for s in scores if s < LOW_CONF_THRESHOLD]
    needs_fallback = len(low_confidence) > LOW_CONF_MAX_COUNT
    if needs_fallback:
        image_b64 = image_to_base64(image)
        completion = client.chat.completions.create(
        model=LLM_VISION_MODEL,
        messages=[
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": """Lies dieses Dokument und gib den exakten Text wieder.
                    Regeln:
                    - Kopiere den Text exakt wie er im Dokument steht
                    - Erfinde keine Wörter oder Sätze
                    - Handgeschriebene Felder mit [HANDSCHRIFT: <text>] markieren
                    - Wenn Text unleserlich ist: [UNLESERLICH]"""}
                ]}
            ]
        )
        if not completion:
            raise ValueError("LLM Vision Model failed to extract text.")
        print(completion.choices[0].message.content)


 Sehr geehrte/r Abnehmer!

In, der Zahl 26 auf der Rechnung steht, habe ich die folgenden Informationen für Sie bereitgestellt:

- Der Betrag beträgt 507,46 Euro.
- Die Zahlungsdatum ist der 13.09.2023.
- Wir haben Ihnen eine Rechnung mit der Nummer 5074610 geschickt.
- Sie können sich auf die folgenden Informationen verlassen:
  - Der Betrag beträgt 507,46 Euro.
  - Die Zahlungsdatum ist der 13.09.2023.
  - Wir haben Ihnen eine Rechnung mit der Nummer 5074610 geschickt.

Bitte befragen Sie sich bei Fragen oder Anregungen zum Zahlungsprozess.

Mit freundlichen Grüßen,
[Name]

---

Sehr geehrte/r Abnehmer!

In, der Zahl 26 auf der Rechnung steht, habe ich die folgenden Informationen für Sie bereitgestellt:

- Der Betrag beträgt 507,46 Euro.
- Die Zahlungsdatum ist der 13.09.2023.
- Wir haben Ihnen eine Rechnung mit der Nummer 5074610 geschickt.
- Sie können sich auf die folgenden Informationen verlassen:
  - Der Betrag beträgt 507,46 Euro.
  - Die Zahlungsdatum ist der 13.09.2023.
  - W

## Vision LLM Evaluation: LLaVA 1.6 Mistral 7B

LLaVA was tested as a fallback for scanned documents with handwritten content.

Results:
- Severe hallucinations: invented text, numbers and dates not present in the document
- Repetition loops: model repeated the same block indefinitely
- Handwritten fields were ignored entirely

Conclusion: LLaVA 7B is a general-purpose vision model, not a document OCR specialist.
It is unsuitable as a reliable fallback for structured document extraction.

## OCR Roadmap

| Approach         | Status      | Notes                                  |
|------------------|-------------|----------------------------------------|
| pdfplumber       | ✅ Production-ready | Digital PDFs                          |
| Tesseract        | ✅ Production-ready | Clean scanned documents               |
| LLaVA 7B         | ❌ Rejected  | Hallucinations, repetition loops      |
| Docling (IBM)    | 🔜 To explore | Designed for document OCR             |
| GPT-4o / Claude  | 🔜 To explore | API-based, best quality, not free     |